In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix,accuracy_score,precision_score,
                             recall_score,f1_score,roc_auc_score)

df=pd.read_csv(r"C:\Users\KSA\Desktop\5th-sem_ojt\project1_diabetes\data\prosessed\najah_p1w2_features.csv")
feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_category","glucose_band"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled,y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)
decision_tree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)



In [3]:
def score_row(name, pred,scores=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found":int(tp),"missed":int(fn),"false_alarm":int(fp)}
    if scores is not None :
        r["auc"]= round(roc_auc_score(y_test,scores),4)
    return r
print("train",x_train.shape,"test:",x_test.shape)
print("diabetic patients int he test set:",int(np.sum(y_test)),"of",len(y_test))
for nm,m,xt in [("logistic regression",logreg,x_test_scaled),
                ("desicion tree (d=3)",decision_tree,x_test),
                ("KNN (k=25)",knn,x_test_scaled)]:
    print(f" {nm:22s}accuracy{accuracy_score(y_test,m.predict(xt)):.4f}")    
          

    
    

train (760, 13) test: (190, 13)
diabetic patients int he test set: 55 of 190
 logistic regression   accuracy0.7526
 desicion tree (d=3)   accuracy0.7211
 KNN (k=25)            accuracy0.7474


In [4]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182 # week 5 logistic regression at threshold 0.30

print("PROJECT METRIC : recall (guard: F1 >=",GUARD_F1,")")
print("TARGET          : recall >=", TARGET_RECALL)
print("BEST SO FAR     :", BEST_SO_FAR, "(Week 5 baseline at threshold 0.30)")
print("GAP TO CLOSE    :", round(TARGET_RECALL - BEST_SO_FAR, 4))

PROJECT METRIC : recall (guard: F1 >= 0.5 )
TARGET          : recall >= 0.7
BEST SO FAR     : 0.6182 (Week 5 baseline at threshold 0.30)
GAP TO CLOSE    : 0.0818


## How an SVM Makes a Decision

An **SVM (Support Vector Machine)** classifies data by finding a boundary between groups.

- **Hyperplane:** The boundary that separates the groups. In a 2D graph, it is a line; in higher dimensions, it is called a hyperplane.
- **Support vectors:** The data points closest to the hyperplane. They help determine where the boundary is placed.
- **Margin:** The distance between the hyperplane and the nearest support vectors. The SVM tries to make this margin as wide as possible.

### Decision process

1. The SVM finds a hyperplane that separates the groups.
2. It uses the support vectors to position the hyperplane.
3. It chooses a hyperplane with a wide margin.
4. For a new data point, it checks which side of the hyperplane the point falls on and predicts that group.

# train an svm with defualt settings

In [5]:
from sklearn.svm import SVC

svm= SVC(random_state=42)
svm.fit(x_train_scaled,y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None


In [6]:
from sklearn.svm import SVC

svm_rbf= SVC(random_state=42)
svm_rbf.fit(x_train_scaled,y_train)
pred_rbf= svm_rbf.predict(x_test_scaled)
print("filtted",svm_rbf)
print("patients flagged:",int(pred_rbf.sum()),"of",len(pred_rbf))

filtted SVC(random_state=42)
patients flagged: 17 of 190


In [7]:
row_rbf = score_row("SVC rbf (defualt)",pred_rbf,
                    svm_rbf.decision_function(x_test_scaled))
print()
print("TARGET RECALL :",TARGET_RECALL,"| best so far :",BEST_SO_FAR)
print("this model    :",row_rbf["recall"])


TARGET RECALL : 0.7 | best so far : 0.6182
this model    : 0.1818


try straight boundry


In [8]:
svm_linear= SVC(kernel="linear",random_state=42)
svm_linear.fit(x_train_scaled,y_train)
pred_linear=svm_linear.predict(x_test_scaled)

row_lin = score_row("SVC linear",pred_linear,
                    svm_linear.decision_function(x_test_scaled))
print(pd.DataFrame([row_lin,row_rbf]).to_string(index=False))





            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
       SVC linear    0.7105     0.0000  0.0000 0.0000      0      55            0 0.6919
SVC rbf (defualt)    0.7263     0.5882  0.1818 0.2778     10      45            7 0.6443


In [9]:
print("patirnts the linear svm flagged as diabetics :",
      int(pred_linear.sum()),"of",len(pred_linear))
print()
print("linear svm confution matrix")
print(confusion_matrix(y_test,pred_linear))
print("dummy model confution matrix")
print(confusion_matrix(y_test,np.zeros_like(y_test)))
print("accuracy score of linear svm :",
      round(accuracy_score(y_test,pred_linear),4))
print("accuracy score of dummy      :",
      round(accuracy_score(y_test,np.zeros_like(y_test)),4))



patirnts the linear svm flagged as diabetics : 0 of 190

linear svm confution matrix
[[135   0]
 [ 55   0]]
dummy model confution matrix
[[135   0]
 [ 55   0]]
accuracy score of linear svm : 0.7105
accuracy score of dummy      : 0.7105


# **learn what class_wieght doews**


`class_weight` gives some classes **more importance** during training.

It is useful when one class has many more examples than another. For example, if 95% of emails are “not spam,” the model might mostly predict “not spam.” Giving the “spam” class a higher weight makes mistakes on spam count more in training.

**In short:** higher class weight means the model is penalized more for misclassifying that class.*


In [10]:
svm_rbf_bal = SVC (random_state=42,class_weight="balanced").fit(x_train_scaled,y_train)
svm_lin_bal =SVC(kernel="linear",random_state=42,class_weight="balanced").fit(x_train_scaled,y_train)

rows= [row_rbf,
       score_row("svc rbf (balanced)", svm_rbf_bal.predict(x_test_scaled),
                 svm_rbf_bal.decision_function(x_test_scaled)),
       row_lin,
        score_row("svc linear (balanced)", svm_lin_bal.predict(x_test_scaled),
                  svm_lin_bal.decision_function(x_test_scaled))]
svm_table = pd.DataFrame(rows)
print(svm_table.to_string(index=False))
      


                model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
    SVC rbf (defualt)    0.7263     0.5882  0.1818 0.2778     10      45            7 0.6443
   svc rbf (balanced)    0.6421     0.4198  0.6182 0.5000     34      21           47 0.6826
           SVC linear    0.7105     0.0000  0.0000 0.0000      0      55            0 0.6919
svc linear (balanced)    0.6579     0.4375  0.6364 0.5185     35      20           45 0.7118


## Get a risk score out of SVM

In [11]:
score_bal = svm_rbf_bal.decision_function(x_test_scaled)
print()
print('decision_function output for the first 8 test patients  ')
print(np.round(score_bal[:8],3))
print()
print('negative = healthy side of the boundary, positive = diabetic side')
print()
print('AUC from decision_function:',
      round(roc_auc_score(y_test, score_bal), 4))
print()
print("AUC of the linear SVM that found nobody:",
      round(roc_auc_score(y_test, svm_linear.decision_function(x_test_scaled)), 4))


decision_function output for the first 8 test patients  
[ 0.027 -1.573 -0.752 -0.552 -0.74  -1.247  0.772  1.099]

negative = healthy side of the boundary, positive = diabetic side

AUC from decision_function: 0.6826

AUC of the linear SVM that found nobody: 0.6919



## check how much of the data the boundary rests on

In [12]:
print('support vectors per class ', svm_rbf.n_support_)
print('total support:', svm_rbf.n_support_.sum(),'of', len(y_train),
      f'= {svm_rbf.n_support_.sum() / len(y_train)*100:.1f}% of the training set')

support vectors per class  [277 212]
total support: 489 of 760 = 64.3% of the training set


## try one change to c

In [13]:
svm_c10 = SVC(random_state=42, C=10).fit(x_train_scaled,y_train)
print(pd.DataFrame([
    row_rbf, 
    score_row("svm_c10", svm_c10.predict(x_test_scaled), svm_c10.decision_function(x_test_scaled))
]).to_string(index=False))

            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
SVC rbf (defualt)    0.7263     0.5882  0.1818 0.2778     10      45            7 0.6443
          svm_c10    0.6684     0.3947  0.2727 0.3226     15      40           23 0.6221


In [14]:
svm_c10 = SVC(random_state=42, C=10).fit(x_train_scaled, y_train)

print(pd.DataFrame([
    row_rbf,
    score_row(
        "SVC rbf C=10",
        svm_c10.predict(x_test_scaled),
        svm_c10.decision_function(x_test_scaled)
    )
]).to_string(index=False))

            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
SVC rbf (defualt)    0.7263     0.5882  0.1818 0.2778     10      45            7 0.6443
     SVC rbf C=10    0.6684     0.3947  0.2727 0.3226     15      40           23 0.6221


## **Explain why many trees beat one**

## Random Forest

- A machine learning model made of many decision trees.
- Each tree makes a prediction.
- The forest combines the trees’ predictions:
  - **Classification:** majority vote
  - **Regression:** average

## Why it can be better than one decision tree

- Usually gives more stable predictions.
- Is less likely to overfit the training data.
- Often predicts more accurately on new data.

**Tradeoff:** It takes more computing power and is harder to interpret than one tree.

## How Random Forest Builds Trees

- **Bootstrap sampling:** Each tree trains on a randomly selected sample of the training data. The sample is drawn *with replacement*, so some rows may appear more than once while others are left out.
- **Random feature selection:** When splitting a tree, the model considers only a random subset of the available features instead of all of them.

These steps make the trees different from one another. The forest combines their predictions, which can make results more stable than relying on a single tree.

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(x_train,y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [16]:
row_rf= score_row("random forest (defualt)",rf.predict(x_test),
                  rf.predict_proba(x_test)[:, 1])

print("trees in the forest : ",rf.n_estimators)
print()
print(pd.DataFrame([row_rf]).to_string(index=False))

trees in the forest :  100

                  model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
random forest (defualt)    0.7105        0.5  0.2545 0.3373     14      41           14 0.7016


In [17]:
print("Random forest (defualt,100 trees)")
print("training accuracy",round(accuracy_score(y_train,rf.predict(x_train)),4))
print("test accuracy",round(accuracy_score(y_test,rf.predict(x_test)),4))
print()
print("week 4 unrestricted single tree")
print("training accuracy :1.0")
print("test accuracy     :0.6474")



Random forest (defualt,100 trees)
training accuracy 1.0
test accuracy 0.7105

week 4 unrestricted single tree
training accuracy :1.0
test accuracy     :0.6474


In [18]:
print("Distinct probabilities - random forest:",
      len(np.unique(rf.predict_proba(x_test)[:,1])))
print("Distinct probabilities - week 4 tree:",
      len(np.unique(decision_tree.predict_proba(x_test)[:,1])))



Distinct probabilities - random forest: 65
Distinct probabilities - week 4 tree: 6


In [19]:
print(f"{'model':28s} {'recall':8s} {'f1':8s}")
print(f"{'Random Forest (default)':28s} {row_rf['recall']:.4f} {row_rf['f1']:.4f}")
print(f"{'best so far(w5 tuned)':28s}{BEST_SO_FAR:8.4f}{0.5397:8.4f}")
print(f"{'TARGET':28s}{TARGET_RECALL:8.4f}{GUARD_F1:8.4f}")
print()
print("closedup gap",row_rf["recall"]>= TARGET_RECALL)


model                        recall   f1      
Random Forest (default)      0.2545 0.3373
best so far(w5 tuned)         0.6182  0.5397
TARGET                        0.7000  0.5000

closedup gap False


Controlling the forest 

In [20]:
rf_bal= RandomForestClassifier(random_state=42,
                               class_weight="balanced").fit(x_train,y_train)
row_rf_bal = score_row(
    "RF balanced",
    rf_bal.predict(x_test),
    rf_bal.predict_proba(x_test)[:,1]
)
print(pd.DataFrame([row_rf,row_rf_bal]).to_string(index=False))

                  model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
random forest (defualt)    0.7105     0.5000  0.2545 0.3373     14      41           14 0.7016
            RF balanced    0.7053     0.4898  0.4364 0.4615     24      31           25 0.6949


Limit the depth of every day

In [21]:
rf_d5  =RandomForestClassifier(random_state=42,max_depth=5).fit(x_train,y_train)
rf_d5_bal  =RandomForestClassifier(random_state=42,max_depth=5,class_weight="balanced").fit(x_train,y_train)

row_rf_d5 = score_row(
    "RF depth = 5",
    rf_d5.predict(x_test),
    rf_d5.predict_proba(x_test)[:,1]
)

row_rf_d5_bal = score_row(
    "RF depth =5 balanced",
    rf_d5_bal.predict(x_test),
    rf_d5_bal.predict_proba(x_test)[:,1]
)


print(pd.DataFrame([row_rf,row_rf_bal,row_rf_d5_bal,row_rf_d5]).to_string(index=False))

                  model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
random forest (defualt)    0.7105     0.5000  0.2545 0.3373     14      41           14 0.7016
            RF balanced    0.7053     0.4898  0.4364 0.4615     24      31           25 0.6949
   RF depth =5 balanced    0.6842     0.4675  0.6545 0.5455     36      19           41 0.7476
           RF depth = 5    0.7421     0.7500  0.1636 0.2687      9      46            3 0.7446


# build the forest variant table

In [22]:
variants = [
    ("RF default (100 trees)",RandomForestClassifier(random_state=42)),
    ("RF balanced", RandomForestClassifier(random_state=42,
                                   class_weight="balanced")),
    ("RF max_depth=5",RandomForestClassifier(random_state=42,max_depth=5 )),
    ("RF max_depth=5 balanced",RandomForestClassifier(random_state=42, max_depth=5,
                                                        class_weight="balanced")),
    ("RF balanced_subsample",RandomForestClassifier( random_state=42,
                                                class_weight="balanced_subsample")),
]
rows = []
for name, m in variants:
    m.fit(x_train, y_train)
    rows.append(score_row(name,m.predict(x_test),m.predict_proba(x_test)[:, 1]))

forest_table = pd.DataFrame(rows)
print(forest_table.to_string(index=False))

best = forest_table.sort_values(
    ["recall", "f1"], ascending=False
).iloc[0]

print("\nBest on the project metric:",best["model"],"| recall:", best["recall"],"| f1:", best["f1"])

rf_best = RandomForestClassifier(random_state=42,max_depth=5,
                                 class_weight="balanced").fit(x_train, y_train)

                  model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
 RF default (100 trees)    0.7105     0.5000  0.2545 0.3373     14      41           14 0.7016
            RF balanced    0.7053     0.4898  0.4364 0.4615     24      31           25 0.6949
         RF max_depth=5    0.7421     0.7500  0.1636 0.2687      9      46            3 0.7446
RF max_depth=5 balanced    0.6842     0.4675  0.6545 0.5455     36      19           41 0.7476
  RF balanced_subsample    0.7316     0.6000  0.2182 0.3200     12      43            8 0.7218

Best on the project metric: RF max_depth=5 balanced | recall: 0.6545 | f1: 0.5455


In [23]:
import time 

t0=time.time()
rf_100=RandomForestClassifier(random_state=42).fit(x_train,y_train)
t_100=time.time() - t0

t0=time.time()
rf_300=RandomForestClassifier(random_state=42,n_estimators=300).fit(x_train,y_train)
t_300=time.time() - t0

print(pd.DataFrame([
    score_row("RF 100",rf_100.predict(x_test),rf_100.predict_proba(x_test)[:,1]),
    score_row("RF 300",rf_300.predict(x_test),rf_300.predict_proba(x_test)[:,1]),
]).to_string(index=False))
print(f"Fitting time: 100 trees {t_100: .2f}s | 300 trees {t_300: .2f}s"
      f" ({t_300 / t_100:.1f}x)")

 model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
RF 100    0.7105     0.5000  0.2545 0.3373     14      41           14 0.7016
RF 300    0.7158     0.5185  0.2545 0.3415     14      41           13 0.7149
Fitting time: 100 trees  0.85s | 300 trees  2.55s (3.0x)


In [24]:
comparison=pd.DataFrame({
    "forest (defualt)":  pd.Series(rf.feature_importances_, index=x.columns),
    "forest (choosen)":  pd.Series(rf_best.feature_importances_, index=x.columns),
    "week 4 single tree":  pd.Series(decision_tree.feature_importances_, index=x.columns),
    }).round(4).sort_values("forest (choosen)",ascending=False)
print(comparison.to_string())

                            forest (defualt)  forest (choosen)  week 4 single tree
glucose                               0.1749            0.2590              0.5927
age                                   0.1409            0.1475              0.2393
diabetes_pedigree                     0.1273            0.1156              0.0472
bmi                                   0.1316            0.1110              0.0000
glucose_band_Normal                   0.0295            0.0944              0.0000
insulin                               0.1042            0.0719              0.0782
blood_pressure                        0.1110            0.0706              0.0000
skin_thickness                        0.0888            0.0570              0.0426
pregnancies                           0.0488            0.0333              0.0000
bmi_category_Obese                    0.0127            0.0166              0.0000
glucose_band_Prediabetic              0.0147            0.0117              0.0000
bmi_

In [25]:
print("top feature share")
print("single tree:%.1f%%" % (comparison["week 4 single tree"].max()*100))

print("choosen forest:%.1f%%" % (comparison["forest (choosen)"].max()*100))
print()
print("feature with exactly zero importance")
print("single tree:",int((comparison["week 4 single tree"]==0).sum()),"of 13")
print("choosen forest:", int((comparison["forest (choosen)"]==0).sum()),"of 13")


top feature share
single tree:59.3%
choosen forest:25.9%

feature with exactly zero importance
single tree: 8 of 13
choosen forest: 0 of 13


# run the week 5 threshols sweep on the forest

In [26]:
proba_rf=rf_best.predict_proba(x_test)[:,1]
print("distinct probabilities from this forest :",len(np.unique(proba_rf)))
print()
sweep= pd.DataFrame([score_row(f"threshold{t:.2f}",(proba_rf >= t ).astype(int))
                     for t in [0.60,0.55,0.50,0.45,0.40,0.35,0.30]])
print(sweep.to_string(index=False))

distinct probabilities from this forest : 190

        model  accuracy  precision  recall     f1  found  missed  false_alarm
threshold0.60    0.7579     0.6216  0.4182 0.5000     23      32           14
threshold0.55    0.7211     0.5161  0.5818 0.5470     32      23           30
threshold0.50    0.6842     0.4675  0.6545 0.5455     36      19           41
threshold0.45    0.6211     0.4190  0.8000 0.5500     44      11           61
threshold0.40    0.5684     0.3866  0.8364 0.5287     46       9           73
threshold0.35    0.5263     0.3723  0.9273 0.5312     51       4           86
threshold0.30    0.4684     0.3487  0.9636 0.5121     53       2           99


# FIND THE SETTING THAT REACHES THE TARGET

In [27]:
hits = sweep [(sweep["recall"] >= TARGET_RECALL) & (sweep["f1"] >= GUARD_F1)]
print("Rows clearing the week 5 (recall >= 0.70 AND f1 >= 0.50) :")
print(hits.to_string(index=False) if len(hits) else"   none")
if len(hits):
    b=hits.sort_values("f1",ascending=False).iloc[0]
    print("\nbest of the them :",b["model"],"|recall",b["recall"],"|f1",b["f1"],
          "| found",int(b["found"]),"of 55")

Rows clearing the week 5 (recall >= 0.70 AND f1 >= 0.50) :
        model  accuracy  precision  recall     f1  found  missed  false_alarm
threshold0.45    0.6211     0.4190  0.8000 0.5500     44      11           61
threshold0.40    0.5684     0.3866  0.8364 0.5287     46       9           73
threshold0.35    0.5263     0.3723  0.9273 0.5312     51       4           86
threshold0.30    0.4684     0.3487  0.9636 0.5121     53       2           99

best of the them : threshold0.45 |recall 0.8 |f1 0.55 | found 44 of 55


# decide which forest goes into the comparison

*it is the best row in the variant table on both the project (recall 0.6545)and the guard (F1 0.5455)
and unlike the 0.45-threshold candidate it has not been tuned against the test set . the honest week6 figure is the one 
that goes forward.*